# ZCU111 ADS-B capture bring-up and baseline experiments

Recorded hardware configuration: ZCU111 with XM500, ADC tile 1 block 0,
2.560 GSPS RF-ADC rate, total decimation by 256, 10 MSPS complex output,
and a nominal centre frequency of 1090 MHz.

The notebook performs the following checks in order:

1. PYNQ environment and overlay-file validation;
2. RF reference-clock setup and programmable-logic download;
3. finite AXI DMA capture with no persistent buffer allocation;
4. time-domain and static spectrum inspection;
5. continuously updating real-time FFT display;
6. baseline sliding-window energy measurement;
7. `.ci16` capture and readback;
8. optional UDP transfer to a host receiver.


## Physical setup record

The default RF input is XM500 J2, mapped to `ADC225_T1_Ch0`. Initial DMA
checks use no applied RF signal or a 50 ohm termination. Tone checks use a
known low-level source near 1090 MHz and remain within the verified input
limits of the board, XM500 path, and any intervening filter or attenuator.


In [ ]:
from pathlib import Path
import importlib
import platform
import sys

# The package may be opened directly or from the Jupyter notebooks root.
search_paths = [
    Path.cwd().resolve(),
    Path.cwd().resolve() / "adsb_capture",
    Path.cwd().resolve() / "pynq" / "adsb_capture",
    Path("/home/xilinx/jupyter_notebooks/adsb_capture"),
]

package_dir = next(
    (path for path in search_paths if (path / "adsb_capture.py").is_file()),
    None,
)
if package_dir is None:
    raise FileNotFoundError(
        "adsb_capture.py was not found in the notebook or board deployment paths"
    )

package_parent = str(package_dir.parent)
sys.path = [entry for entry in sys.path if entry != package_parent]
sys.path.insert(0, package_parent)
for module_name in list(sys.modules):
    if module_name == "adsb_capture" or module_name.startswith("adsb_capture."):
        del sys.modules[module_name]
importlib.invalidate_caches()

bitfile = package_dir / "bitstream" / "adsb_capture.bit"
hwhfile = package_dir / "bitstream" / "adsb_capture.hwh"

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Package directory:", package_dir)
print("Bitstream present:", bitfile.is_file(), bitfile)
print("HWH present:", hwhfile.is_file(), hwhfile)

if not bitfile.is_file() or not hwhfile.is_file():
    raise FileNotFoundError(
        "A matched adsb_capture.bit and adsb_capture.hwh pair is required"
    )


## PYNQ environment

The runtime requires `pynq`, `xrfdc`, and `xrfclk`. The installed PYNQ image
provides these packages on the ZCU111.


In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pynq as pynq_runtime
import xrfdc
import xrfclk
from IPython.display import display

import adsb_capture as adsb_capture_package
from adsb_capture import (
    AdsbCapture,
    IQ_SAMPLE_RATE_HZ,
    PL_DECIMATION,
    PL_DECIMATION_SELECT,
    RFDC_DECIMATION,
    RFDC_FABRIC_WORDS,
    unpack_iq,
)

print("PYNQ:", pynq_runtime.__version__)
print("Loaded adsb_capture:", adsb_capture_package.__file__)
print("Complex sample rate:", f"{IQ_SAMPLE_RATE_HZ / 1e6:.3f} MSPS")
print("RFDC decimation:", RFDC_DECIMATION)
print("RFDC words per beat:", RFDC_FABRIC_WORDS)
print("PL decimation:", PL_DECIMATION)
print("PL selector:", PL_DECIMATION_SELECT)


## Overlay initialisation

`AdsbCapture` reads the HWH metadata, programs the RF reference clocks to
409.6 MHz, downloads the bitstream, resolves the named IP blocks, configures
the RFDC mixer for decimation by 8 with two words per beat and selects
programmable-logic decimation by 32 (selector 5).


In [ ]:
CENTRE_FREQUENCY_MHZ = 1090.0

load_started = time.monotonic()
radio = AdsbCapture(
    bitfile=str(bitfile),
    centre_frequency_mhz=CENTRE_FREQUENCY_MHZ,
)
load_elapsed = time.monotonic() - load_started

required_ip = {
    "rfdc",
    "decimator",
    "axi_dma",
    "capture_control",
    "capture_status",
}
available_ip = set(radio.overlay.ip_dict)
missing_ip = required_ip - available_ip
if missing_ip:
    raise RuntimeError(f"Required HWH instances are absent: {sorted(missing_ip)}")

print("Overlay load time:", f"{load_elapsed:.2f} s")
print("Centre frequency:", f"{radio.centre_frequency_mhz:.3f} MHz")
print("RFDC status:", radio.adc_block.BlockStatus)
print("RFDC runtime decimation:", radio.adc_block.DecimationFactor)
print("RFDC runtime words/beat:", radio.adc_block.FabRdVldWords)
print("PL selector readback:", radio.decimator.read(0x00))
print("Hardware build ID:", f"0x{radio.hardware_build_id:04X}")
print("RFDC fabric clock:", f"{radio.fabric_clock_hz / 1e6:.3f} MHz")
print("Output TVALID interval:", radio.output_valid_interval, "fabric clocks")
print("Required IP instances: present")


## DMA capture helper

The returned PYNQ DMA buffer is released after conversion. The resulting
NumPy array is independent of the DMA allocation and uses complex64 values
normalised by 32768.


In [ ]:
def acquire_iq(sample_count):
    """Return one normalised complex64 frame with no retained DMA buffer."""
    buffer = radio.capture(int(sample_count))
    try:
        return unpack_iq(buffer, normalize=True)
    finally:
        buffer.freebuffer()


## Baseline DMA capture

A 4096-sample frame provides the initial no-input DMA check. Successful
completion confirms the basic capture gate, FIFO, clock converter, DMA, and
PYNQ control path.


In [ ]:
smoke_iq = acquire_iq(4096)
smoke_rms = np.sqrt(np.mean(np.abs(smoke_iq) ** 2))

print("Shape:", smoke_iq.shape)
print("Data type:", smoke_iq.dtype)
print("Mean I:", float(smoke_iq.real.mean()))
print("Mean Q:", float(smoke_iq.imag.mean()))
print("Normalised RMS magnitude:", float(smoke_rms))
print("Peak magnitude:", float(np.max(np.abs(smoke_iq))))


## Working capture

The default working frame contains 262,144 samples, corresponding to
26.2144 ms at 10 MSPS. This frame is reused by the time-domain, spectrum,
and energy experiments below.


In [ ]:
CAPTURE_SAMPLES = 262_144

capture_started = time.monotonic()
iq = acquire_iq(CAPTURE_SAMPLES)
capture_elapsed = time.monotonic() - capture_started
duration_s = len(iq) / IQ_SAMPLE_RATE_HZ

print("Complex samples:", len(iq))
print("Signal duration:", f"{duration_s * 1e3:.4f} ms")
print("Capture call duration:", f"{capture_elapsed * 1e3:.2f} ms")


## Time-domain inspection

The first 200 microseconds are plotted as normalised I, Q, and magnitude.


In [ ]:
plot_samples = min(len(iq), int(200e-6 * IQ_SAMPLE_RATE_HZ))
time_us = np.arange(plot_samples) / IQ_SAMPLE_RATE_HZ * 1e6

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(time_us, iq.real[:plot_samples], label="I", linewidth=0.8)
axes[0].plot(time_us, iq.imag[:plot_samples], label="Q", linewidth=0.8)
axes[0].set_ylabel("Normalised amplitude")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(time_us, np.abs(iq[:plot_samples]), linewidth=0.8)
axes[1].set_xlabel("Time (us)")
axes[1].set_ylabel("Magnitude")
axes[1].grid(True)
fig.tight_layout()


## Frequency-domain inspection

DC is removed before applying a Hann window. Frequency is shown both as an
offset from the configured centre frequency and as absolute RF frequency. A
1090 MHz test tone is expected near zero offset. A 1090.5 MHz test tone is
expected approximately 0.5 MHz from zero; the observed sign records spectral
orientation.


In [ ]:
spectrum_input = iq - iq.mean()
window = np.hanning(len(spectrum_input))
spectrum = np.fft.fftshift(np.fft.fft(spectrum_input * window))
offset_hz = np.fft.fftshift(
    np.fft.fftfreq(len(spectrum_input), d=1 / IQ_SAMPLE_RATE_HZ)
)
magnitude_db = 20 * np.log10(np.abs(spectrum) + 1e-12)
peak_index = int(np.argmax(magnitude_db))
peak_offset_hz = float(offset_hz[peak_index])
absolute_frequency_hz = radio.centre_frequency_mhz * 1e6 + offset_hz

print("Largest FFT-bin offset:", f"{peak_offset_hz / 1e6:.6f} MHz")
print(
    "Largest FFT-bin RF frequency:",
    f"{(radio.centre_frequency_mhz * 1e6 + peak_offset_hz) / 1e6:.6f} MHz",
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(offset_hz / 1e6, magnitude_db, linewidth=0.8)
axes[0].set_xlabel("Offset from centre frequency (MHz)")
axes[0].set_ylabel("Magnitude (dB)")
axes[0].grid(True)

axes[1].plot(absolute_frequency_hz / 1e6, magnitude_db, linewidth=0.8)
axes[1].set_xlabel("RF frequency (MHz)")
axes[1].set_ylabel("Magnitude (dB)")
axes[1].grid(True)
fig.tight_layout()


## Real-time FFT

This display repeatedly captures a DMA frame, applies a Hann window, and
updates an exponentially averaged spectrum. The default run lasts 30
seconds; set `LIVE_FFT_SECONDS = None` to continue until the notebook cell
is interrupted. Because the overlay uses finite simple-DMA captures, this
is a live display with short gaps between frames rather than a gap-free
streaming spectrum analyser.


In [ ]:
LIVE_FFT_SAMPLES = 65_536
LIVE_FFT_SECONDS = 30.0  # Use None to run until interrupted.
LIVE_FFT_REFRESH_HZ = 5.0
LIVE_FFT_AVERAGING = 0.65
LIVE_FFT_FLOOR_DBFS = -120.0


def run_live_fft(seconds=LIVE_FFT_SECONDS):
    window = np.hanning(LIVE_FFT_SAMPLES).astype(np.float32)
    window_gain = float(window.sum())
    offset_hz = np.fft.fftshift(
        np.fft.fftfreq(LIVE_FFT_SAMPLES, d=1 / IQ_SAMPLE_RATE_HZ)
    )
    rf_frequency_mhz = (
        radio.centre_frequency_mhz * 1e6 + offset_hz
    ) / 1e6

    fig, axis = plt.subplots(figsize=(12, 5))
    line, = axis.plot(
        rf_frequency_mhz,
        np.full(LIVE_FFT_SAMPLES, LIVE_FFT_FLOOR_DBFS),
        linewidth=0.8,
    )
    axis.set_xlim(rf_frequency_mhz[0], rf_frequency_mhz[-1])
    axis.set_ylim(LIVE_FFT_FLOOR_DBFS, 5.0)
    axis.set_xlabel("RF frequency (MHz)")
    axis.set_ylabel("Magnitude (dBFS)")
    axis.grid(True)
    fig.tight_layout()
    display_handle = display(fig, display_id=True)

    averaged_power = None
    frame_count = 0
    started = time.monotonic()
    refresh_period = 1.0 / max(float(LIVE_FFT_REFRESH_HZ), 0.1)

    try:
        while seconds is None or time.monotonic() - started < float(seconds):
            frame_started = time.monotonic()
            live_iq = acquire_iq(LIVE_FFT_SAMPLES)
            live_iq = live_iq - live_iq.mean()
            spectrum = np.fft.fftshift(np.fft.fft(live_iq * window))
            power = (np.abs(spectrum) / window_gain) ** 2

            if averaged_power is None:
                averaged_power = power
            else:
                averaged_power = (
                    LIVE_FFT_AVERAGING * averaged_power
                    + (1.0 - LIVE_FFT_AVERAGING) * power
                )

            magnitude_dbfs = 10.0 * np.log10(
                np.maximum(averaged_power, 1e-20)
            )
            peak_index = int(np.argmax(magnitude_dbfs))
            peak_frequency_mhz = float(rf_frequency_mhz[peak_index])
            peak_dbfs = float(magnitude_dbfs[peak_index])

            line.set_ydata(magnitude_dbfs)
            axis.set_title(
                f"Live FFT — peak {peak_frequency_mhz:.6f} MHz, "
                f"{peak_dbfs:.1f} dBFS"
            )
            display_handle.update(fig)
            frame_count += 1

            remaining = refresh_period - (time.monotonic() - frame_started)
            if remaining > 0:
                time.sleep(remaining)
    except KeyboardInterrupt:
        print("Live FFT stopped by user.")
    finally:
        elapsed = time.monotonic() - started
        display_handle.update(fig)
        plt.close(fig)
        print(
            f"Displayed {frame_count} FFT frames in {elapsed:.2f} s "
            f"({frame_count / max(elapsed, 1e-9):.2f} frames/s)."
        )


run_live_fft()


## Baseline energy experiment

A 0.5 microsecond moving average is applied to magnitude-squared samples.
The displayed threshold is the median plus eight scaled median absolute
deviations. This is a diagnostic threshold rather than a validated ADS-B
detector. The plot is centred on the maximum-energy sample.


In [ ]:
energy_window_samples = max(1, round(0.5e-6 * IQ_SAMPLE_RATE_HZ))
power = np.abs(iq) ** 2
kernel = np.ones(energy_window_samples) / energy_window_samples
moving_energy = np.convolve(power, kernel, mode="same")

energy_median = np.median(moving_energy)
energy_mad = np.median(np.abs(moving_energy - energy_median)) + 1e-15
energy_threshold = energy_median + 8.0 * 1.4826 * energy_mad
above_threshold = np.flatnonzero(moving_energy > energy_threshold)

peak_sample = int(np.argmax(moving_energy))
half_span = int(100e-6 * IQ_SAMPLE_RATE_HZ)
start = max(0, peak_sample - half_span)
stop = min(len(iq), peak_sample + half_span)
local_time_us = (np.arange(start, stop) - peak_sample) / IQ_SAMPLE_RATE_HZ * 1e6

print("Moving-average samples:", energy_window_samples)
print("Threshold:", float(energy_threshold))
print("Samples above threshold:", len(above_threshold))
print("Maximum-energy sample index:", peak_sample)

plt.figure(figsize=(12, 4))
plt.plot(local_time_us, moving_energy[start:stop], linewidth=0.8)
plt.axhline(energy_threshold, color="red", linestyle="--", label="Threshold")
plt.xlabel("Time from maximum-energy sample (us)")
plt.ylabel("Moving-average energy")
plt.grid(True)
plt.legend()


## `.ci16` capture and readback

A separate one-million-sample frame is stored under
`/home/xilinx/captures`. The raw file contains little-endian signed int16
values ordered as `I0,Q0,I1,Q1,...`. The adjacent JSON file records capture
parameters.


In [ ]:
SAVE_SAMPLES = 1_000_000
capture_directory = Path("/home/xilinx/captures")
capture_name = time.strftime("adsb_%Y%m%d_%H%M%S.ci16", time.gmtime())
data_path, metadata_path = radio.save(
    str(capture_directory / capture_name),
    samples=SAVE_SAMPLES,
)

metadata = json.loads(metadata_path.read_text())
raw = np.fromfile(data_path, dtype="<i2")
if raw.size % 2:
    raise ValueError("Stored ci16 file contains an incomplete I/Q pair")
disk_iq = (
    raw[0::2].astype(np.float32)
    + 1j * raw[1::2].astype(np.float32)
) / 32768.0

print("Data file:", data_path)
print("Metadata file:", metadata_path)
print("File size:", data_path.stat().st_size, "bytes")
print("Readback samples:", len(disk_iq))
print(json.dumps(metadata, indent=2))


## Optional UDP transfer

The host receiver starts before the board sender:

```bash
python3 receive_iq_udp.py host_capture.ci16 \
  --bind 0.0.0.0 --port 50000 --frames 2
```

The transfer cell remains disabled by default. `UDP_HOST` records the host
interface reachable from the ZCU111.


In [ ]:
RUN_UDP_TEST = False
UDP_HOST = "192.168.2.1"
UDP_PORT = 50000
UDP_FRAMES = 2
UDP_FRAME_SAMPLES = 65_536

if RUN_UDP_TEST:
    radio.stream_udp(
        UDP_HOST,
        port=UDP_PORT,
        frame_samples=UDP_FRAME_SAMPLES,
        frames=UDP_FRAMES,
    )
    print("UDP frames sent:", UDP_FRAMES)
else:
    print("UDP test disabled; set RUN_UDP_TEST = True after the host receiver starts")


## Recorded values for repeated runs

The principal comparison values are RFDC sampling frequency, centre
frequency, normalised RMS and peak magnitude, largest static/live FFT-bin
offset, energy threshold, number of samples above threshold, capture
duration, and dropped
UDP packet count. RF source level, cable path, filter, attenuator, and XM500
connector are recorded alongside these values for controlled comparisons.
